# DSA 4060 – Week 9 Guided Lab
## From Implicit Events to Ranked Recommendations with Confidence-Weighted ALS

**Course:** DSA 4060 – Recommender Systems  
**Environment:** Visual Studio Code + Jupyter Notebook  
**Main output:** A trained implicit-feedback ALS model and Top-5 recommendations.

### What this notebook does
This notebook creates a small, reproducible music-streaming event log. It converts plays, repeat plays, and saves into numerical interaction strengths, prepares a sparse user–item matrix, trains an Alternating Least Squares (ALS) model, and recommends five tracks to a selected user.



## 1. Learning objectives

By the end of the lab, you should be able to:

1. Explain the difference between **explicit feedback** and **implicit feedback**.
2. Convert observed events into interaction strength.
3. Explain **preference** and **confidence** in an implicit-feedback recommender.
4. Build a sparse user–item matrix.
5. Train an implicit-feedback ALS model.
6. Generate and critique a ranked Top-5 recommendation list.
7. Change one model setting and compare the results.
8. Describe how a recommender should be evaluated using later interactions.

### How to work through this notebook
Run the cells **from top to bottom**, in order. Read each Markdown section before running the code below it. At each checkpoint, pause and answer the questions in your own words.

## 2. Environment setup

This project uses:

- **Python** – the programming language.
- **NumPy** – numerical operations and random data generation.
- **pandas** – event logs and result tables.
- **SciPy** – sparse matrix storage.
- **implicit** – the ALS recommendation algorithm.

### VS Code setup
Open the project folder in VS Code. Make sure the Microsoft **Python** and **Jupyter** extensions are installed. Select the Python kernel for this project from the notebook's top-right **Select Kernel** menu.

If you have not installed the packages yet, open the VS Code terminal and run:

```bash
python -m pip install -r requirements.txt
```

You can also use the next cell to install packages into the active notebook kernel. If you use the terminal installation, you may skip the installation cell.

In [1]:
%pip install implicit pandas numpy scipy

   ---------------------------------------- 0.0/676.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/676.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/676.7 kB ? eta -:--:--
   --------------- ------------------------ 262.1/676.7 kB ? eta -:--:--
   ---------------------------------------- 676.7/676.7 kB 1.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import the libraries

The imports make the tools available to the notebook:

- `numpy` creates arrays and reproducible random values.
- `pandas` stores and analyses the event data.
- `csr_matrix` creates a memory-efficient sparse matrix.
- `AlternatingLeastSquares` is the ALS model implementation from the `implicit` package.

A fixed random seed helps the synthetic dataset and model demonstration remain reproducible.

In [2]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

np.random.seed(4060)

c:\Users\HALIMA MAHDI\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Create a small event log

An **event log** records actions that users performed. In this music example, the actions are:

- `play`: the user played a track.
- `repeat_play`: the user played a track again.
- `save`: the user saved a track.

Each row represents one observed event. It is **not** a star rating.

The code creates 12 users and 18 tracks. Each user is assigned six tracks, and each selected track receives between one and four event records.

In [3]:
users = [f"U{i:02d}" for i in range(1, 13)]
items = [f"Track_{i:02d}" for i in range(1, 19)]
event_types = ["play", "repeat_play", "save"]

rows = []

for user in users:
    preferred = np.random.choice(items, size=6, replace=False)
    for item in preferred:
        n = np.random.randint(1, 5)
        for _ in range(n):
            rows.append({
                "user_id": user,
                "item_id": item,
                "event_type": np.random.choice(
                    event_types,
                    p=[0.60, 0.25, 0.15]
                )
            })

events = pd.DataFrame(rows)

print("First five events:")
display(events.head())

print("Dataset dimensions (rows, columns):", events.shape)

First five events:


,user_id,item_id,event_type
0,U01,Track_02,repeat_play
1,U01,Track_02,repeat_play
2,U01,Track_02,play
3,U01,Track_02,save
4,U01,Track_06,repeat_play


Dataset dimensions (rows, columns): (194, 3)


### Inspect the event log

These summaries help you understand the generated data before modelling it.

- `value_counts()` counts how many times each event type appears.
- `groupby('user_id').size()` counts the event rows for each user.
- `describe()` gives summary statistics for those counts.

**Checkpoint 1 – What do the rows mean?**

1. What might a repeat play or a save suggest about a user's interest?
2. What can we *not* safely conclude from those actions?
3. What biases could appear in synthetic data or in a real platform's event log?

Remember: a user may replay a track because it was enjoyable, because it autoplayed, or because it was played accidentally. An absent event does not automatically mean dislike.

In [4]:
print("Event counts:")
display(events["event_type"].value_counts())

print("Number of event rows per user:")
display(events.groupby("user_id").size().describe())

Event counts:


event_type
play           120
repeat_play     44
save            30
Name: count, dtype: int64

Number of event rows per user:


count    12.000000
mean     16.166667
std       3.511885
min      12.000000
25%      13.750000
50%      15.000000
75%      17.750000
max      23.000000
dtype: float64

## 4. Convert events into interaction strength

Different actions can be assigned different numerical weights. For this demonstration:

| Event | Weight | Reason for the example |
|---|---:|---|
| Play | 1.0 | Basic observed engagement |
| Repeat play | 3.0 | Repeated engagement |
| Save | 5.0 | A stronger intentional action |

These values are **modelling choices**, not universal truths. A save is not always worth exactly five plays. A real system should test alternative weights and consider accidental clicks, autoplay, and repeated exposure.

When a user performs several events on the same track, the code adds their weights. The resulting `strength` is the total interaction strength for that user–track pair.

In [5]:
event_weight = {
    "play": 1.0,
    "repeat_play": 3.0,
    "save": 5.0
}

events["weight"] = events["event_type"].map(event_weight)

interactions = (
    events
    .groupby(["user_id", "item_id"], as_index=False)["weight"]
    .sum()
    .rename(columns={"weight": "strength"})
)

display(interactions.sort_values("strength", ascending=False).head(10))

,user_id,item_id,strength
1,U01,Track_02,12.0
71,U12,Track_12,12.0
61,U11,Track_03,12.0
69,U12,Track_09,12.0
14,U03,Track_11,11.0
70,U12,Track_10,11.0
8,U02,Track_10,10.0
11,U02,Track_17,10.0
67,U12,Track_05,10.0
54,U10,Track_06,10.0


**Checkpoint 2 – Challenge the weights**

1. Should one save always be worth five plays? Explain your view.
2. How could accidental clicks, autoplay, or repeated exposure distort interaction strength?
3. Which alternative weights would you test first, and why?

## 5. Calculate preference and confidence

Implicit-feedback ALS distinguishes between two ideas:

### Preference
In this lab, preference is binary:

- `1` means an interaction was observed.
- `0` means no positive interaction was observed.

For the aggregated interaction rows, strength is positive, so their preference is `1`.

### Confidence
Confidence indicates how strongly the model should trust an observed interaction. The lab uses:

\[
c_{ui} = 1 + \alpha r_{ui}
\]

where:

- \(c_{ui}\) is confidence for user \(u\) and item \(i\),
- \(lpha\) is a scaling parameter,
- \(r_{ui}\) is the observed interaction strength.

Here, `alpha = 20`. A larger strength therefore produces a larger confidence value.

**Important:** The confidence column is calculated here to make the idea visible. The `implicit` ALS library also receives the raw strength matrix and applies its `alpha` setting during model fitting.

In [6]:
alpha = 20

interactions["preference"] = (interactions["strength"] > 0).astype(int)
interactions["confidence"] = 1 + alpha * interactions["strength"]

display(
    interactions[
        ["user_id", "item_id", "strength", "preference", "confidence"]
    ].head(10)
)

,user_id,item_id,strength,preference,confidence
0,U01,Track_01,2.0,1,41.0
1,U01,Track_02,12.0,1,241.0
2,U01,Track_06,6.0,1,121.0
3,U01,Track_08,3.0,1,61.0
4,U01,Track_11,8.0,1,161.0
5,U01,Track_13,6.0,1,121.0
6,U02,Track_03,3.0,1,61.0
7,U02,Track_04,3.0,1,61.0
8,U02,Track_10,10.0,1,201.0
9,U02,Track_15,1.0,1,21.0


### Compare two examples

For `alpha = 20`:

- If strength = 1, confidence = \(1 + 20(1) = 21\).
- If strength = 12, confidence = \(1 + 20(12) = 241\).

Both cases have preference `1`, because an interaction was observed. The second case has higher confidence because its observed strength is larger.

**Checkpoint 3 – Compare two cases**

1. How do preference and confidence differ for strength 1 versus strength 12?
2. Why should unobserved user–item pairs have low confidence rather than be treated as definite dislikes?
3. What might happen if `alpha` is extremely large?

In [7]:
example_strengths = pd.DataFrame({"strength": [1, 12]})
example_strengths["preference"] = (example_strengths["strength"] > 0).astype(int)
example_strengths["confidence"] = 1 + alpha * example_strengths["strength"]
display(example_strengths)

,strength,preference,confidence
0,1,1,21
1,12,1,241


## 6. Build the sparse user–item matrix

ALS needs interactions in matrix form.

- Each **row** represents one user.
- Each **column** represents one item (track).
- A populated cell contains that user's interaction strength for that track.
- A missing/zero cell means no interaction was recorded in this dataset. It does not prove that the user dislikes the track.

The code first creates mappings between readable IDs and integer positions. It then creates a **CSR (Compressed Sparse Row) matrix** using SciPy.

A sparse matrix stores the non-zero values efficiently. This is useful because, in a large recommender system, most users interact with only a small fraction of all available items.

In [8]:
user_ids = sorted(interactions["user_id"].unique())
item_ids = sorted(interactions["item_id"].unique())

user_to_idx = {user: index for index, user in enumerate(user_ids)}
item_to_idx = {item: index for index, item in enumerate(item_ids)}
idx_to_item = {index: item for item, index in item_to_idx.items()}

row = interactions["user_id"].map(user_to_idx)
col = interactions["item_id"].map(item_to_idx)
data = interactions["strength"].astype(float)

user_item = csr_matrix(
    (data, (row, col)),
    shape=(len(user_ids), len(item_ids))
)

print("Matrix shape (users, items):", user_item.shape)
print("Observed user-item pairs:", user_item.nnz)
print("Matrix density:", user_item.nnz / (user_item.shape[0] * user_item.shape[1]))

Matrix shape (users, items): (12, 18)
Observed user-item pairs: 72
Matrix density: 0.3333333333333333


**Checkpoint 4 – Read the matrix**

1. What does one row represent? What does one column represent?
2. Why is a sparse matrix more appropriate than a dense table for a large catalogue?
3. What does matrix density tell you about how much of the user–item space has observed interactions?

## 7. Train an implicit-feedback ALS model

**ALS** stands for Alternating Least Squares. It learns compact numerical representations (called **latent factors**) for users and items.

The model alternates between:
1. Learning user factors while item factors are held fixed.
2. Learning item factors while user factors are held fixed.

The user and item factors occupy the same latent space. Their relationship helps the model estimate which items may be relevant to each user.

Model settings used in this lab:

- `factors=16`: number of latent dimensions.
- `regularization=0.10`: helps reduce overfitting by penalising overly large factor values.
- `alpha=20`: confidence scaling used by the implicit-feedback model.
- `iterations=25`: number of ALS training iterations.
- `random_state=4060`: makes the demonstration more reproducible.

In [9]:
model = AlternatingLeastSquares(
    factors=16,
    regularization=0.10,
    alpha=alpha,
    iterations=25,
    random_state=4060
)

model.fit(user_item)

print("User factor matrix shape:", model.user_factors.shape)
print("Item factor matrix shape:", model.item_factors.shape)

c:\Users\HALIMA MAHDI\AppData\Local\Programs\Python\Python312\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 25/25 [00:00<00:00, 1337.50it/s]

User factor matrix shape: (12, 16)
Item factor matrix shape: (18, 16)


**Checkpoint 5 – Connect code to concepts**

1. What does `factors=16` control?
2. Why might regularization help?
3. Why is a fixed random seed useful during a class demonstration?

## 8. Generate Top-5 recommendations

The model can produce a ranked list of items for a selected user. We use `U01` as the example.

`filter_already_liked_items=True` asks the model to leave out items that already appear in the user's observed interaction history. The model returns item indices and scores; the mapping converts indices back into readable track IDs.

**Do not interpret the scores as probabilities.** They are ranking scores produced by the model. A score of 0.8 does not mean an 80% chance that the user will like the track.

In [10]:
target_user = "U01"
u = user_to_idx[target_user]

item_idx, scores = model.recommend(
    u,
    user_item[u],
    N=5,
    filter_already_liked_items=True
)

recommendations = pd.DataFrame({
    "item_id": [idx_to_item[index] for index in item_idx],
    "score": scores
})

seen = (
    interactions.loc[
        interactions["user_id"].eq(target_user),
        ["item_id", "strength"]
    ]
    .sort_values("strength", ascending=False)
)

print("Observed history for", target_user)
display(seen.head(10))

print("Recommended next (Top-5)")
display(recommendations)

Observed history for U01


,item_id,strength
1,Track_02,12.0
4,Track_11,8.0
5,Track_13,6.0
2,Track_06,6.0
3,Track_08,3.0
0,Track_01,2.0


Recommended next (Top-5)


,item_id,score
0,Track_05,0.186877
1,Track_04,0.125515
2,Track_09,0.093726
3,Track_07,0.069388
4,Track_12,0.063325


**Checkpoint 6 – Critique the ranking**

1. Do the recommended tracks look plausible given the user's observed history?
2. Why do model scores not behave like probabilities?
3. What real-world constraints might require reranking? Consider diversity, freshness, availability, and safety.

## 9. Run a one-variable experiment

A fair experiment changes **one setting at a time**. Keep the other settings the same so that you can discuss which change may have affected the ranking.

Choose one experiment:

| Experiment | Values to compare | Question |
|---|---|---|
| Confidence scale | `alpha = 5` versus `alpha = 40` | Does stronger confidence change the ranking? |
| Latent dimension | `factors = 4` versus `factors = 32` | Does more model capacity produce a different list? |
| Regularization | `0.01` versus `1.0` | Does stronger regularization change or stabilise the ranking? |

The example below compares a second model using `alpha = 5`. It leaves the other settings unchanged. You can edit the value to test a different setting, but change only one parameter per experiment.

In [11]:
experiment_alpha = 5  # Change this one value for your experiment.

experiment_model = AlternatingLeastSquares(
    factors=16,
    regularization=0.10,
    alpha=experiment_alpha,
    iterations=25,
    random_state=4060
)

experiment_model.fit(user_item)

experiment_item_idx, experiment_scores = experiment_model.recommend(
    u,
    user_item[u],
    N=5,
    filter_already_liked_items=True
)

experiment_recommendations = pd.DataFrame({
    "item_id": [idx_to_item[index] for index in experiment_item_idx],
    "score": experiment_scores
})

comparison = pd.DataFrame({
    "baseline_item": recommendations["item_id"],
    "experiment_item": experiment_recommendations["item_id"]
})

print("Baseline Top-5:")
display(recommendations)

print(f"Experiment Top-5 (alpha={experiment_alpha}):")
display(experiment_recommendations)

print("Position-by-position comparison:")
display(comparison)

100%|██████████| 25/25 [00:00<00:00, 2373.53it/s]

Baseline Top-5:


,item_id,score
0,Track_05,0.186877
1,Track_04,0.125515
2,Track_09,0.093726
3,Track_07,0.069388
4,Track_12,0.063325


Experiment Top-5 (alpha=5):


,item_id,score
0,Track_12,0.045923
1,Track_05,0.037284
2,Track_07,0.031147
3,Track_09,0.030300
4,Track_04,0.028581


Position-by-position comparison:


,baseline_item,experiment_item
0,Track_05,Track_12
1,Track_04,Track_05
2,Track_09,Track_07
3,Track_07,Track_09
4,Track_12,Track_04


### Interpret your experiment

Write down:

- Which parameter did you change?
- What value did you use before and after?
- Did any recommended items change position or disappear?
- Why might the parameter have caused that change?
- Did the result improve the recommendations? What evidence would you need before making that claim?

A changed ranking is not automatically a better ranking.

## 10. Evaluation: design before calculating

A model that fits observed events is not automatically useful. A realistic offline evaluation asks whether the model can rank later relevant items highly.

A time-aware evaluation generally follows these steps:

1. Sort each user's events by timestamp. Real event logs need timestamps for this.
2. Use earlier interactions for training.
3. Hold out one or more later interactions as relevant test items.
4. Generate Top-K recommendations without already-seen items.
5. Compute ranking metrics such as Precision@K, Recall@K, or NDCG@K.
6. Compare the model with a simple popularity baseline.

**Limitation of this lab:** The generated event log does not contain timestamps, so this notebook describes the time-aware evaluation design but does not claim to calculate a valid time-based test score.

### Evaluation discussion

1. Why is a random row split risky for event logs?
2. When might Recall@K matter more than Precision@K?
3. What offline result would still not prove that users benefit from the recommender?

## 11. Whole-lab summary

| Concept | Where it appeared | Meaning |
|---|---|---|
| Implicit feedback | Event log and interaction strength | Observed behaviour is useful, but it is not the same as an explicit rating. |
| Confidence weighting | `alpha` and interaction strength | Stronger observed signals receive greater influence. |
| Sparse matrix | CSR user–item matrix | Observed entries are stored efficiently; most pairs are unobserved. |
| ALS | Model training | The model learns compact user and item representations. |
| Ranking | Top-5 output | The practical output is an ordered shortlist, not just a score. |
| Evaluation | Time-aware holdout plan | We test whether future relevant items appear near the top. |

### Exit ticket (about 150–200 words)

Write a short reflection that answers all four prompts:

1. Name one assumption made when converting events into weights.
2. Explain preference versus confidence using one concrete example.
3. Describe one change that altered—or did not alter—your Top-5 ranking.
4. Identify one limitation of this lab model and one next improvement.

## 12. Optional extensions

If you want to develop the lab further:

- Replace the synthetic data with a small real event log containing timestamps.
- Add item metadata and compare collaborative ALS with a feature-aware approach such as LightFM.
- Add a popularity baseline and report Recall@5 and NDCG@5.
- Inspect recommendation coverage and popularity bias across all users.

Do not describe synthetic results as real-world findings. Clearly identify the source and limitations of any real dataset you introduce.